# 999: Combine the Player-Stats Datasets into One Table

This notebook merges the outputs of notebooks 01-03 into **one player-stats table**, with one row per player + season. It gets left-joined onto the salary table (`salary_features_combined.csv`) on `season` + `nba_player_id` before XGBoost.

| Input (`data-files/02-processed/`) | From | Row = |
|---|---|---|
| `llimllib_player_regular_season_features.csv` | 01 | player + season: regular-season stats, age, experience, 3-year averages |
| `llimllib_player_playoff_features.csv` | 02 | player + postseason: playoff stats |
| `bbref_player_bpm_features.csv` | 03 | player + season: regular-season and playoff Box +/-, 3-year Box +/- average |

**Output:** `data-files/02-processed/player_stats_combined.csv`

*Run order: `01` → `02` → `03` → `999`.*

## Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..') / 'salary-data'))
from nba_cleaning_utils import PROCESSED, FIRST_SEASON, LAST_SEASON

ids = {'nba_player_id': str}
rs = pd.read_csv(PROCESSED / 'llimllib_player_regular_season_features.csv', dtype=ids)
po = pd.read_csv(PROCESSED / 'llimllib_player_playoff_features.csv', dtype=ids)
bpm = pd.read_csv(PROCESSED / 'bbref_player_bpm_features.csv', dtype=ids)

print('regular-season rows (01):', rs.shape)
print('playoff rows (02):       ', po.shape)
print('Box +/- rows (03):       ', bpm.shape)

regular-season rows (01): (5968, 22)
playoff rows (02):        (2405, 16)
Box +/- rows (03):        (5975, 7)


## 1. Choose the base table

The base is **every player-season that appears in 01 or 02**: every player who played a regular-season or playoff game. Almost all are in 01. The few playoff-only players (who joined a team after the regular season, notebook 02 section 2B) are added from 02, so their playoff stats aren't lost. Players who didn't make the playoffs keep blank playoff columns. That isn't missing data, since they had no playoff games. 03's Box +/- columns are then **left-joined** onto this base.

## 2. Join keys

All three tables use `season` (end year) + `nba_player_id` (NBA.com id; 03 translated Basketball-Reference's ids). Check that the key is unique in each table.

In [2]:
key = ['season', 'nba_player_id']
for name, t in [('01', rs), ('02', po), ('03', bpm)]:
    print(f'{name}: duplicate keys =', t.duplicated(key).sum())

01: duplicate keys = 0
02: duplicate keys = 0
03: duplicate keys = 0


## 3. Merge

In [3]:
shared = ['player_name', 'name_key', 'last_team', 'age', 'years_in_nba']
merged = rs.merge(po, on=key, how='outer', suffixes=('', '_po'), validate='one_to_one', indicator='source')
print(merged['source'].map({'both': 'regular season + playoffs', 'left_only': 'regular season only',
                            'right_only': 'playoffs only'}).value_counts().to_string())

# Before filling the shared columns from 02, check the two notebooks agree where both have the player
both = merged['source'] == 'both'
for col in shared:
    a, b = merged.loc[both, col], merged.loc[both, f'{col}_po']
    set_both = a.notna() & b.notna()
    print(f'{col:13s} agrees on {(a[set_both] == b[set_both]).mean():.1%} of rows where both have a value'
          f' | blank in 01 only: {(a.isna() & b.notna()).sum()} | blank in 02 only: {(a.notna() & b.isna()).sum()}')

source
regular season only          3570
regular season + playoffs    2398
playoffs only                   7
player_name   agrees on 100.0% of rows where both have a value | blank in 01 only: 0 | blank in 02 only: 0
name_key      agrees on 100.0% of rows where both have a value | blank in 01 only: 0 | blank in 02 only: 0
last_team     agrees on 100.0% of rows where both have a value | blank in 01 only: 0 | blank in 02 only: 74
age           agrees on 100.0% of rows where both have a value | blank in 01 only: 0 | blank in 02 only: 0
years_in_nba  agrees on 100.0% of rows where both have a value | blank in 01 only: 0 | blank in 02 only: 0


In [4]:
for col in shared:
    merged[col] = merged[col].fillna(merged[f'{col}_po'])
merged = merged.drop(columns=[f'{c}_po' for c in shared])
merged['played_in_playoffs'] = merged['source'] != 'left_only'

merged = merged.merge(bpm, on=key, how='left', validate='one_to_one', indicator='bpm_source')
print('rows:', len(merged), '| with a Box +/- row:', (merged['bpm_source'] == 'both').sum())

rows: 5975 | with a Box +/- row: 5975


## 4. Check the merge

**(a) Do 01 and 03 count the same prior seasons for the 3-year averages?** Both use the same window. llimllib counts regular-season rows from llimllib and Basketball-Reference counts its own, so they should agree if the two sources list the same seasons for each player.

In [5]:
has_rs = merged['rs_gp'].notna()
agree = merged.loc[has_rs, 'seasons_played_last_3'] == merged.loc[has_rs, 'bpm_seasons_last_3']
print(f'prior-season counts agree for {agree.mean():.2%} of {has_rs.sum():,} regular-season rows')
merged.loc[has_rs & ~agree, ['season', 'player_name', 'seasons_played_last_3', 'bpm_seasons_last_3']]

prior-season counts agree for 100.00% of 5,968 regular-season rows


,season,player_name,seasons_played_last_3,bpm_seasons_last_3


**(b) Does each player's Box +/- go with his other stats?** Players who matter more to their teams should have a higher BPM, so BPM should rise with minutes per game, and stars should be near the top.

In [6]:
regular = merged[merged['rs_gp'] >= 20]
print('correlation of rs_bpm with rs_mpg (20+ games):', round(regular['rs_bpm'].corr(regular['rs_mpg']), 2))
print('correlation of rs_bpm with rs_ppg (20+ games):', round(regular['rs_bpm'].corr(regular['rs_ppg']), 2))
print('correlation of rs_bpm with on-court rs_plus_minus (20+ games):', round(regular['rs_bpm'].corr(regular['rs_plus_minus']), 2))
regular[regular['season'] == 2026].nlargest(5, 'rs_bpm')[['player_name', 'last_team', 'rs_ppg', 'rs_bpm', 'rs_plus_minus']].round(1)

correlation of rs_bpm with rs_mpg (20+ games): 0.55
correlation of rs_bpm with rs_ppg (20+ games): 0.67
correlation of rs_bpm with on-court rs_plus_minus (20+ games): 0.55


,player_name,last_team,rs_ppg,rs_bpm,rs_plus_minus
5970,Nikola Jokić,DEN,27.7,14.2,8.5
5461,Shai Gilgeous-Alexander,OKC,31.1,11.7,11.6
5705,Victor Wembanyama,SAS,25.0,10.7,10.7
5951,Giannis Antetokounmpo,MIL,27.6,9.5,2.7
5482,Luka Dončić,LAL,33.5,9.3,2.9


**What we found:**

- **01 and 02 agree** on every shared column (name, age, experience, last team) wherever both have a value, so filling the playoff-only players' columns from 02 is safe. The only gaps are 74 rows where 02's `last_team` is blank and 01 has it. We keep 01's value.
- **The prior-season counts from llimllib and Basketball-Reference agree on every regular-season row (100%)**, so the BPM average and the other 3-year averages use the same seasons for every player.
- **BPM behaves as expected next to the other stats.** Among players with 20+ games, it correlates 0.67 with points per game and 0.55 with minutes, and the top five in 2025-26 are Jokić, Gilgeous-Alexander, Wembanyama, Antetokounmpo, and Dončić. Its correlation with on-court plus/minus is only 0.55. That is the point of adding it: Antetokounmpo and Dončić have a BPM above 9 but on-court plus/minus below 3, because on-court plus/minus mostly measures the team.

## 5. Final columns

Keep the keys and the features, in the feature list's order: regular season, playoffs, age and experience, 3-year averages. Drop the columns that were only used for checking (`source`, `bpm_source`, `bpm_seasons_last_3`, `bbref_id`).

In [7]:
final_cols = [
    # keys
    'season', 'nba_player_id', 'player_name', 'name_key', 'last_team',
    # regular season
    'rs_gp', 'rs_mpg', 'rs_ppg', 'rs_rpg', 'rs_apg', 'rs_bpg', 'rs_spg', 'rs_ts_pct', 'rs_bpm', 'rs_plus_minus',
    # playoffs
    'played_in_playoffs', 'po_gp', 'po_mpg', 'po_ppg', 'po_rpg', 'po_apg', 'po_bpg', 'po_spg', 'po_ts_pct', 'po_bpm', 'po_plus_minus',
    # player
    'age', 'years_in_nba',
    # 3-year averages (the three regular seasons before this one)
    'seasons_played_last_3', 'avg_ppg_last_3', 'avg_rpg_last_3', 'avg_apg_last_3', 'avg_bpm_last_3',
    'avg_mpg_last_3', 'avg_gp_last_3',
]
player_stats = (merged[final_cols].astype({'years_in_nba': int}).sort_values(key).reset_index(drop=True))

ps = player_stats
checks = {
    'one row per season + player': not ps.duplicated(key).any(),
    f'seasons are {FIRST_SEASON}-{LAST_SEASON}': ps['season'].between(FIRST_SEASON, LAST_SEASON).all() and ps['season'].nunique() == 11,
    'every row has a name, age, and years in the NBA': ps[['player_name', 'name_key', 'age', 'years_in_nba']].notna().all().all(),
    'every row has regular-season or playoff games': (ps['rs_gp'].notna() | ps['po_gp'].notna()).all(),
    'playoff columns are filled exactly for playoff players': (ps['po_gp'].notna() == ps['played_in_playoffs']).all(),
    'every player with games has a Box +/- (except the one 0-minute row)': (ps['rs_gp'].notna() & ps['rs_bpm'].isna()).sum() + (ps['po_gp'].notna() & ps['po_bpm'].isna()).sum() <= 1,
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
assert all(checks.values())

PASS - one row per season + player
PASS - seasons are 2016-2026
PASS - every row has a name, age, and years in the NBA
PASS - every row has regular-season or playoff games
PASS - playoff columns are filled exactly for playoff players
PASS - every player with games has a Box +/- (except the one 0-minute row)


In [8]:
missing = ps.isna().sum()
missing[missing > 0].rename('missing values').to_frame()

,missing values
last_team,31
rs_gp,7
rs_mpg,7
rs_ppg,7
rs_rpg,7
rs_apg,7
rs_bpg,7
rs_spg,7
rs_ts_pct,36
rs_bpm,7


**Expected missing values:**

- **`po_*`**: players who didn't play in the playoffs (most rows), which is not missing data. `played_in_playoffs` says which rows these are.
- **`rs_*`, `seasons_played_last_3`, and the 3-year averages for the 7 playoff-only players**, who have no regular-season row that season. (Their 3-year averages could exist, but notebook 01 only builds them for regular-season rows. These players are late-season signings who almost never appear in the salary table.)
- **`avg_*_last_3`**: players with no regular-season row in the three previous seasons (`seasons_played_last_3` = 0), mostly rookies.
- **`rs_ts_pct` / `po_ts_pct`**: players with no shot attempts (29 regular-season and 66 playoff rows, on top of the rows above), where true shooting is undefined.
- **`last_team`**: the 31 rows where the source didn't record a team. It isn't a join key.

Leave these as `NaN`. XGBoost learns which way to send missing values at each split.

In [9]:
player_stats.drop(columns='season').describe().T[['count', 'mean', 'min', '50%', 'max']].round(2)

,count,mean,min,50%,max
rs_gp,5968.0,47.11,1.00,53.00,84.00
rs_mpg,5968.0,19.47,0.35,19.16,43.52
rs_ppg,5968.0,8.64,0.00,7.03,36.13
rs_rpg,5968.0,3.54,0.00,3.06,15.99
rs_apg,5968.0,1.95,0.00,1.35,11.74
rs_bpg,5968.0,0.39,0.00,0.28,3.83
rs_spg,5968.0,0.63,0.00,0.57,3.01
rs_ts_pct,5939.0,0.54,0.00,0.55,1.50
rs_bpm,5968.0,-1.54,-54.40,-1.30,240.80
rs_plus_minus,5968.0,-0.42,-30.67,-0.44,28.00


## 6. Save

In [10]:
out = PROCESSED / 'player_stats_combined.csv'
player_stats.to_csv(out, index=False)
print('saved', player_stats.shape, 'to', out)
player_stats.head()

saved (5975, 35) to ..\..\data-files\02-processed\player_stats_combined.csv


,season,nba_player_id,player_name,name_key,last_team,rs_gp,rs_mpg,rs_ppg,rs_rpg,rs_apg,...,po_plus_minus,age,years_in_nba,seasons_played_last_3,avg_ppg_last_3,avg_rpg_last_3,avg_apg_last_3,avg_bpm_last_3,avg_mpg_last_3,avg_gp_last_3
0,2016,101106,Andrew Bogut,andrew bogut,GSW,70.0,20.732286,5.357143,7.028571,2.314286,...,1.545455,31.0,11,3.0,6.484764,8.591884,2.150653,2.466554,24.860806,55.333333
1,2016,101107,Marvin Williams,marvin williams,CHA,81.0,28.866235,11.703704,6.432099,1.358025,...,-9.571429,30.0,11,3.0,7.926792,4.537355,1.172888,-0.349467,25.034233,72.333333
2,2016,101108,Chris Paul,chris paul,LAC,74.0,32.705923,19.540541,4.189189,9.972973,...,7.500000,31.0,11,3.0,18.376310,4.216935,10.199592,8.330613,34.405272,71.333333
3,2016,101109,Raymond Felton,raymond felton,DAL,80.0,27.399938,9.537500,3.225000,3.550000,...,-11.400000,32.0,11,3.0,9.119207,2.260087,4.166589,-0.566168,24.916843,54.000000
4,2016,101111,Charlie Villanueva,charlie villanueva,DAL,62.0,10.726398,5.145161,2.451613,0.387097,...,-2.750000,31.0,11,3.0,5.888330,2.506960,0.459828,0.300000,11.801084,51.000000


**Output:** `data-files/02-processed/player_stats_combined.csv`, **one row per player + season** (2015-16 to 2025-26).

| Feature | Column(s) |
|---|---|
| Regular season: PPG, RPG, APG, BPG, SPG, TS%, games, minutes, Box +/- | `rs_ppg`, `rs_rpg`, `rs_apg`, `rs_bpg`, `rs_spg`, `rs_ts_pct`, `rs_gp`, `rs_mpg`, `rs_bpm` |
| Playoffs: PPG, RPG, APG, BPG, SPG, Box +/-, TS%, minutes | `po_ppg`, `po_rpg`, `po_apg`, `po_bpg`, `po_spg`, `po_bpm`, `po_ts_pct`, `po_mpg` (also `po_gp`, `played_in_playoffs`) |
| Age, years in the NBA | `age`, `years_in_nba` |
| 3-year trend: PPG, RPG, APG, Box +/-, minutes, games played | `avg_ppg_last_3`, `avg_rpg_last_3`, `avg_apg_last_3`, `avg_bpm_last_3`, `avg_mpg_last_3`, `avg_gp_last_3` (+ `seasons_played_last_3`) |
| (extra) on-court plus/minus per game | `rs_plus_minus`, `po_plus_minus` |

**How to join it with the other datasets:** **left-join from the salary table** (`salary_features_combined.csv`) on `season` (end year) + `nba_player_id`. A player traded mid-season has one row here (his full-season stats) but one salary row per team, so each of those rows gets his full-season stats. Don't join on `last_team`. For a name-based table, use `season` + `name_key`. Because we only predict offseason trades, each season's stats are final when the trade window opens.